# Pipeline de Avaliação — TCC Willian Fernandes Dias — PUC-SP 2026
## Tabelas 4 (BLEU), 5 (IFC) e 6 (RTT) — Corpus exato dos notebooks originais

**Modelos:** M2M100 · T5 (Flan-T5-large) · NLLB-200 · mBART-50  
**Tipos textuais:** Texto Papal (Dei Verbum) · Notícia (Corriere) · Poema (L'Infinito) · Canção (Bella Ciao)  
**Par:** Italiano → Português

---
### Rastreabilidade dos corpora
- **Bella Ciao:** 20 versos — extraídos de `comparativo_bella_ciao_4_modelos.ipynb`
- **Notícia:** 14 segmentos — extraídos de `modelo_comparativo_dos_4_modelos_em_1_noticia.ipynb`
- **Poema:** 15 versos — extraídos de `Cópia_de_modelo_comparativo_dos_4_modelos_usando_um_poema.ipynb`
- **Texto Papal:** 26 segmentos — Dei Verbum §§1–26 (corpus expandido para execução real)

> **Requisito:** Runtime GPU (T4). Vá em Ambiente de execução → Alterar tipo → GPU T4

In [ ]:
# Célula 1 — Instalação
!pip install -q sacrebleu transformers sentencepiece torch accelerate pandas tabulate

In [ ]:
# Célula 2 — Imports
import torch, re, statistics
import numpy as np
import pandas as pd
from tabulate import tabulate
import sacrebleu
from sacrebleu.metrics import BLEU
from transformers import (
    M2M100ForConditionalGeneration, M2M100Tokenizer,
    AutoModelForSeq2SeqLM, AutoTokenizer,
    MBartForConditionalGeneration, MBart50TokenizerFast,
)
import warnings; warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
bleu_sent = BLEU(effective_order=True)

In [ ]:
# Célula 3 — Corpora exatos dos notebooks originais

# ── BELLA CIAO (20 versos) ────────────────────────────────────────────────────
# Fonte: comparativo_bella_ciao_4_modelos.ipynb
BELLA_CIAO_IT = [
    "Una mattina mi son' svegliato,",
    "O bella ciao, bella ciao, bella ciao ciao ciao,",
    "Una mattina mi son' svegliato",
    "E ho trovato l'invasor.",
    "O partigiano, portami via,",
    "O bella ciao, bella ciao, bella ciao ciao ciao,",
    "O partigiano, portami via,",
    "Ché mi sento di morir.",
    "E se io muoio da partigiano,",
    "O bella ciao, bella ciao, bella ciao ciao ciao,",
    "E se io muoio da partigiano,",
    "Tu mi devi seppellir.",
    "E seppellire lassù in montagna,",
    "O bella ciao, bella ciao, bella ciao ciao ciao,",
    "E seppellire lassù in montagna",
    "Sotto l'ombra di un bel fior.",
    "E quest'è il fiore del partigiano,",
    "O bella ciao, bella ciao, bella ciao ciao ciao,",
    "E quest'è il fiore del partigiano",
    "Morto per la libertà.",
]
BELLA_CIAO_PT = [
    "Uma manhã eu me levantei,",
    "Ó bella ciao, bella ciao, bella ciao ciao ciao,",
    "Uma manhã eu me levantei",
    "E encontrei o invasor.",
    "Ó partigiano, leva-me contigo,",
    "Ó bella ciao, bella ciao, bella ciao ciao ciao,",
    "Ó partigiano, leva-me contigo,",
    "Que me sinto a morrer.",
    "E se eu morrer como partigiano,",
    "Ó bella ciao, bella ciao, bella ciao ciao ciao,",
    "E se eu morrer como partigiano,",
    "Você me deve enterrar.",
    "E me enterrar lá no alto da montanha,",
    "Ó bella ciao, bella ciao, bella ciao ciao ciao,",
    "E me enterrar lá no alto da montanha",
    "À sombra de uma bela flor.",
    "E esta é a flor do partigiano,",
    "Ó bella ciao, bella ciao, bella ciao ciao ciao,",
    "E esta é a flor do partigiano",
    "Morto pela liberdade.",
]

# ── NOTÍCIA (14 segmentos) ────────────────────────────────────────────────────
# Fonte: modelo_comparativo_dos_4_modelos_em_1_noticia.ipynb
NOTICIA_IT = [
    "Come fa l'Iran a resistere a Usa e Israele (e a bucare le loro difese)? La strategia dietro la «non sconfitta» di Teheran",
    "Quello dell'Iran è un piano dichiarato. Il suo obiettivo è un conflitto d'attrito che riesca a spaventare la Casa Bianca, i mercati, l'elettorato. Le sue capacità si sono rivelate superiori alle previsioni. Ma sta subendo danni consistenti",
    "Sono due i messaggi che arrivano dal Golfo: l'Iran ha mantenuto le sue promesse belliche ed ha adattato il proprio dispositivo. Non sono sorprese ma le conferme di quanto era stato messo in conto. È questa la valutazione del momento di un conflitto che nessuno sa come possa concludersi.",
    "LE ULTIME NOTIZIE SULLA GUERRA IN IRAN, IN DIRETTA",
    "Nelle settimane precedenti a Epic Fury, Teheran aveva spiegato quale sarebbe stata la sua risposta ad un eventuale assalto avversario.",
    "Inoltre, sempre gli iraniani, avevano insistito su un punto: Donald Trump non deve illudersi su uno scontro limitato.",
    "È un piano dichiarato che punta ad un conflitto d'attrito, una guerra continua che deve spaventare la Casa Bianca, i mercati, l'elettorato.",
    "La strategia di Teheran è il risultato di esperienze disseminate nel tempo. Ha dimostrato capacità di adattamento: dagli Houthi, a Hezbollah, ad Hamas.",
    "L'arsenale missilistico e le falangi – a basso costo – dei droni kamikaze hanno consentito di portare avanti la battaglia.",
    "Gli sciami di proiettili hanno costretto Usa, Israele e gli alleati a impiegare un alto numero di intercettori.",
    "Proprio il Pentagono, nell'imminenza dell'assalto, aveva messo in guardia su ciò poteva accadere.",
    "Per contro l'offensiva congiunta dell'IDF e del Pentagono ha debilitato ulteriormente le difese iraniane.",
    "C'è, infine, un fronte sospeso e riguarda un passaggio geografico fondamentale, lo Stretto di Hormuz.",
    "Per i commentatori forse è una carta di riserva tenuta nel caso l'Arabia Saudita decidesse di uscire dal patto di non coinvolgimento.",
]
NOTICIA_PT = [
    "Como o Irã consegue resistir aos EUA e a Israel (e perfurar suas defesas)? A estratégia por trás da 'não derrota' de Teerã",
    "O plano do Irã é declarado. Seu objetivo é um conflito de desgaste que consiga assustar a Casa Branca, os mercados e o eleitorado. Suas capacidades se revelaram superiores às previsões. Mas está sofrendo danos consistentes.",
    "Há duas mensagens chegando do Golfo: o Irã manteve suas promessas de guerra e adaptou seu dispositivo. Não são surpresas, mas confirmações do que já havia sido previsto. Esta é a avaliação atual de um conflito que ninguém sabe como pode terminar.",
    "ÚLTIMAS NOTÍCIAS SOBRE A GUERRA NO IRÃ, AO VIVO",
    "Nas semanas que antecederam a Epic Fury, Teerã tinha explicado qual seria a sua resposta a um eventual ataque adversário.",
    "Além disso, os iranianos também tinham insistido num ponto: Donald Trump não se deve iludir com um confronto limitado.",
    "É um plano declarado que aponta para um conflito de desgaste, uma guerra contínua que deve assustar a Casa Branca, os mercados e o eleitorado.",
    "A estratégia de Teerã é o resultado de experiências disseminadas ao longo do tempo. Demonstrou capacidade de adaptação: dos Houthis ao Hezbollah, ao Hamas.",
    "O arsenal de mísseis e as falanges – de baixo custo – de drones kamikazes permitiram levar a batalha adiante.",
    "Os enxames de projéteis obrigaram os EUA, Israel e os aliados a empregar um elevado número de interceptadores.",
    "O próprio Pentágono, na iminência do ataque, tinha alertado para o que poderia acontecer.",
    "Em contrapartida, a ofensiva conjunta das FDI e do Pentágono debilitou ainda mais as defesas iranianas.",
    "Há, por fim, uma frente suspensa que diz respeito a uma passagem geográfica fundamental, o Estreito de Ormuz.",
    "Para os comentaristas, talvez seja um trunfo na manga guardado para o caso de a Arábia Saudita decidir sair do pacto de não envolvimento.",
]

# ── POEMA — L'Infinito (15 versos) ───────────────────────────────────────────
# Fonte: Cópia_de_modelo_comparativo_dos_4_modelos_usando_um_poema.ipynb
POEMA_IT = [
    "Sempre caro mi fu quest'ermo colle,",
    "E questa siepe, che da tanta parte",
    "Dell'ultimo orizzonte il guardo esclude.",
    "Ma sedendo e mirando, interminati",
    "Spazi di là da quella, e sovrumani",
    "Silenzi, e profondissima quiete",
    "Io nel pensier mi fingo; ove per poco",
    "Il cor non si spaura. E come il vento",
    "Odo stormir tra queste piante, io quello",
    "Infinito silenzio a questa voce",
    "Vo comparando: e mi sovvien l'eterno,",
    "E le morte stagioni, e la presente",
    "E viva, e il suon di lei. Cosi tra questa",
    "Immensita s'annega il pensier mio:",
    "E il naufragar m'è dolce in questo mare.",
]
POEMA_PT = [
    "Sempre me foi caro este ermo monte,",
    "E esta sebe, que de tanta parte",
    "Do último horizonte exclui o olhar.",
    "Mas sentando e contemplando, intermináveis",
    "Espaços além dela, e sobre-humanos",
    "Silêncios, e profundíssima quiete",
    "Eu na mente imagino; onde por pouco",
    "O coração não se apavora. E como o vento",
    "Ouço sussurrar entre essas plantas, eu aquele",
    "Infinito silêncio a esta voz",
    "Vou comparando: e me vem à mente o eterno,",
    "E as mortas estações, e a presente",
    "E viva, e o seu som. Assim entre esta",
    "Imensidão se afoga o pensamento meu:",
    "E o naufragar me é doce neste mar.",
]

# ── TEXTO PAPAL — Dei Verbum (26 segmentos) ──────────────────────────────────
# Corpus expandido para execução real dos modelos
PAPAL_IT = [
    "Il sacro Concilio, udendo con religiosa attenzione la parola di Dio e proclamandola con ferma fiducia, fa sue queste parole di san Giovanni.",
    "Piacque a Dio nella sua bontà e sapienza rivelare se stesso e far conoscere il mistero della sua volontà.",
    "Con questa rivelazione, Dio invisibile nel suo immenso amore parla agli uomini come ad amici e si intrattiene con essi.",
    "Dopo aver a più riprese e in più modi parlato per mezzo dei profeti, Dio ultimamente ha parlato a noi per mezzo del Figlio.",
    "Gesù Cristo, Verbo fatto carne, mandato come uomo agli uomini, parla le parole di Dio.",
    "A Dio che rivela è dovuta l'obbedienza della fede.",
    "Con la fede l'uomo si abbandona interamente a Dio liberamente.",
    "La sacra Tradizione e la sacra Scrittura sono strettamente connesse e comunicanti tra loro.",
    "La sacra Tradizione e la sacra Scrittura costituiscono un solo sacro deposito della parola di Dio affidato alla Chiesa.",
    "L'ufficio d'interpretare autenticamente la parola di Dio scritta o trasmessa è affidato al Magistero.",
    "Il Magistero non è superiore alla parola di Dio, ma la serve insegnando solo ciò che è stato trasmesso.",
    "È chiaro dunque che la sacra Tradizione, la sacra Scrittura e il Magistero della Chiesa sono tra loro strettamente congiunti.",
    "Le verità divinamente rivelate, che nei libri della sacra Scrittura sono contenute ed espresse, furono scritte per ispirazione dello Spirito Santo.",
    "Per la composizione dei libri sacri, Dio scelse degli uomini di cui si servì nell'esercizio delle loro facoltà e capacità naturali.",
    "Poiché Dio nella sacra Scrittura ha parlato per mezzo di uomini alla maniera degli uomini, l'interprete deve ricercare il senso inteso dagli agiografi.",
    "La sacra Scrittura deve essere letta e interpretata con lo stesso Spirito con il quale è stata scritta.",
    "La parola di Dio, che è potenza di Dio per la salvezza di chiunque crede, si trova nei libri del Nuovo Testamento.",
    "I Vangeli hanno un'origine apostolica e occupano a buon diritto un posto preminente tra tutti i Libri sacri.",
    "La formazione del Vangelo avvenne in tre tappe: la vita e il magistero di Gesù, la predicazione degli apostoli, e la redazione scritta.",
    "Il Canone dei Libri Sacri comprende anche gli scritti apostolici che attestano la presenza operante di Cristo.",
    "La Chiesa ha sempre venerato le divine Scritture come venera lo stesso Corpo di Cristo.",
    "Nei libri sacri il Padre che è nei cieli viene con molta amorevolezza incontro ai suoi figli e discorre con essi.",
    "La sacra teologia si basa, come su un fondamento perenne, sulla parola di Dio scritta, insieme con la sacra Tradizione.",
    "I traduttori devono rendere accessibile la parola di Dio nelle lingue di tutti i popoli con le necessarie correzioni.",
    "Occorre che i fedeli abbiano largo accesso alla sacra Scrittura per nutrirsene e rafforzarsi nella vita cristiana.",
    "La Chiesa, sposa del Verbo incarnato, mediante la sacra Scrittura progredisce nella comprensione dei sacri testi.",
]
PAPAL_PT = [
    "O sacro Concílio, ouvindo religiosamente a Palavra de Deus e proclamando-a com firme confiança, faz suas estas palavras de São João.",
    "Aprouve a Deus na sua bondade e sabedoria revelar-se a si e dar a conhecer o mistério da sua vontade.",
    "Com esta revelação, Deus invisível no seu imenso amor fala aos homens como amigos e se entretém com eles.",
    "Depois de ter falado por meio dos profetas, muitas vezes e em diversos modos, Deus ultimamente falou a nós por meio do Filho.",
    "Jesus Cristo, Verbo feito carne, enviado como homem aos homens, fala as palavras de Deus.",
    "À obediência da fé o homem deve-se a Deus que revela.",
    "Com a fé, o homem se abandona todo a Deus livremente.",
    "A sacra Tradição e a sacra Escritura estão estritamente conectadas e comunicantes entre elas.",
    "A sacra Tradição e a sacra Escritura constituem um só depósito sagrado da Palavra de Deus, confiado à Igreja.",
    "O encargo de interpretar autenticamente a Palavra de Deus escrita ou transmitida foi confiado ao Magistério.",
    "O Magistério não é superior à Palavra de Deus, mas a serve ensinando somente o que foi transmitido.",
    "É claro, pois, que a sacra Tradição, a sacra Escritura e o Magistério da Igreja estão estreitamente unidos entre si.",
    "As verdades divinamente reveladas, que nos livros da sacra Escritura são contidas e expressas, foram escritas por inspiração do Espírito Santo.",
    "Para a composição dos livros sacros, Deus escolheu homens, dos quais se serviu no exercício das suas faculdades e capacidades naturais.",
    "Visto que Deus na sacra Escritura falou por meio de homens à maneira humana, o intérprete deve pesquisar o sentido visado pelos hagiógrafos.",
    "A sacra Escritura deve ser lida e interpretada com o mesmo Espírito pelo qual foi escrita.",
    "A Palavra de Deus, que é força de Deus para a salvação de todo crente, encontra-se nos livros do Novo Testamento.",
    "Os Evangelhos têm origem apostólica e ocupam com razão lugar preeminente entre todos os Livros sagrados.",
    "A formação do Evangelho ocorreu em três etapas: a vida e o magistério de Jesus, a pregação dos apóstolos e a redação escrita.",
    "O Cânon dos Livros Sagrados inclui também os escritos apostólicos que atestam a presença operante de Cristo.",
    "A Igreja sempre venerou as divinas Escrituras como venera o próprio Corpo de Cristo.",
    "Nos livros sacros, o Pai que está nos céus vem com muita amorosidade ao encontro dos seus filhos e com eles conversa.",
    "A sagrada teologia apoia-se como num fundamento perene na Palavra de Deus escrita, juntamente com a sacra Tradição.",
    "Os tradutores devem tornar acessível a Palavra de Deus nas línguas de todos os povos com as necessárias correções.",
    "É necessário que os fiéis tenham amplo acesso à sacra Escritura para nela se nutrirem e fortalecerem na vida cristã.",
    "A Igreja, esposa do Verbo encarnado, por meio da sacra Escritura progride na compreensão dos textos sagrados.",
]

# ── Termos culturais para IFC ─────────────────────────────────────────────────
TERMOS_IFC = {
    'Bella Ciao':   [('bella ciao','bella ciao'),('partigiano','partigiano'),
                     ('liberdade','libertà'),('invasor','invasor'),('montanha','montagna')],
    'Notícia':      [('Irã','Iran'),('Teerã','Teheran'),('pasdaran','pasdaran'),
                     ('Pentágono','Pentagono'),('Hormuz','Hormuz')],
    'Poema':        [('ermo','ermo'),('infinito','infinito'),('monte','colle'),
                     ('sebe','siepe'),('naufragar','naufragar')],
    'Texto Papal':  [('Revelação','Rivelazione'),('Tradição','Tradizione'),
                     ('Escritura','Scrittura'),('Magistério','Magistero'),
                     ('Concílio','Concilio')],
}

CORPORA = {
    'Bella Ciao':  {'it': BELLA_CIAO_IT, 'ref': BELLA_CIAO_PT},
    'Notícia':     {'it': NOTICIA_IT,    'ref': NOTICIA_PT},
    'Poema':       {'it': POEMA_IT,      'ref': POEMA_PT},
    'Texto Papal': {'it': PAPAL_IT,      'ref': PAPAL_PT},
}

print('Corpus carregado:')
for nome, c in CORPORA.items():
    print(f'  {nome}: {len(c["it"])} segmentos, {len(TERMOS_IFC[nome])} termos IFC')

In [ ]:
# Célula 4 — Carregar os 4 modelos
# ⏱ ~5–10 min com GPU

MODELOS = {}

# M2M100
print('Carregando M2M100...')
tok = M2M100Tokenizer.from_pretrained('facebook/m2m100_418M')
mod = M2M100ForConditionalGeneration.from_pretrained('facebook/m2m100_418M',
      torch_dtype=torch.float16 if DEVICE=='cuda' else torch.float32).to(DEVICE)
tok.src_lang = 'it'
MODELOS['M2M100'] = {'tok': tok, 'mod': mod,
                      'fn': lambda t, tok=tok, mod=mod: _gen_m2m(t, tok, mod)}

# T5
print('Carregando T5...')
from transformers import T5ForConditionalGeneration, T5Tokenizer
tok5 = T5Tokenizer.from_pretrained('google/flan-t5-large')
mod5 = T5ForConditionalGeneration.from_pretrained('google/flan-t5-large',
       torch_dtype=torch.float16 if DEVICE=='cuda' else torch.float32).to(DEVICE)
MODELOS['T5'] = {'tok': tok5, 'mod': mod5,
                  'fn': lambda t, tok=tok5, mod=mod5: _gen_t5(t, tok, mod)}

# NLLB-200
print('Carregando NLLB-200...')
tokn = AutoTokenizer.from_pretrained('facebook/nllb-200-distilled-600M', src_lang='ita_Latn')
modn = AutoModelForSeq2SeqLM.from_pretrained('facebook/nllb-200-distilled-600M',
       torch_dtype=torch.float16 if DEVICE=='cuda' else torch.float32).to(DEVICE)
tgt_nllb = tokn.convert_tokens_to_ids('por_Latn')
MODELOS['NLLB'] = {'tok': tokn, 'mod': modn,
                    'fn': lambda t, tok=tokn, mod=modn, tgt=tgt_nllb: _gen_nllb(t, tok, mod, tgt)}

# mBART
print('Carregando mBART...')
tokm = MBart50TokenizerFast.from_pretrained('facebook/mbart-large-50-many-to-many-mmt')
modm = MBartForConditionalGeneration.from_pretrained('facebook/mbart-large-50-many-to-many-mmt',
       torch_dtype=torch.float16 if DEVICE=='cuda' else torch.float32).to(DEVICE)
tokm.src_lang = 'it_IT'
tgt_mb = tokm.lang_code_to_id['pt_XX']
MODELOS['mBART'] = {'tok': tokm, 'mod': modm,
                     'fn': lambda t, tok=tokm, mod=modm, tgt=tgt_mb: _gen_mbart(t, tok, mod, tgt)}

print('\n✓ 4 modelos carregados')

In [ ]:
# Célula 5 — Funções de geração por modelo

def _gen_m2m(texto, tok, mod):
    enc = tok(texto, return_tensors='pt', max_length=512, truncation=True).to(DEVICE)
    with torch.no_grad():
        out = mod.generate(**enc, forced_bos_token_id=tok.get_lang_id('pt'),
                           max_new_tokens=512, num_beams=4)
    return tok.decode(out[0], skip_special_tokens=True)

def _gen_t5(texto, tok, mod):
    entrada = 'Translate from Italian to Portuguese: ' + texto
    enc = tok(entrada, return_tensors='pt', max_length=512, truncation=True).to(DEVICE)
    with torch.no_grad():
        out = mod.generate(**enc, max_new_tokens=512, num_beams=4)
    return tok.decode(out[0], skip_special_tokens=True)

def _gen_nllb(texto, tok, mod, tgt_id):
    enc = tok(texto, return_tensors='pt', max_length=512, truncation=True).to(DEVICE)
    with torch.no_grad():
        out = mod.generate(**enc, forced_bos_token_id=tgt_id,
                           max_new_tokens=512, num_beams=4)
    return tok.decode(out[0], skip_special_tokens=True)

def _gen_mbart(texto, tok, mod, tgt_id):
    enc = tok(texto, return_tensors='pt', max_length=512, truncation=True).to(DEVICE)
    with torch.no_grad():
        out = mod.generate(**enc, forced_bos_token_id=tgt_id,
                           max_new_tokens=512, num_beams=4)
    return tok.decode(out[0], skip_special_tokens=True)

def traduzir_corpus(segmentos, fn_modelo, nome_mod, nome_tipo):
    trad = []
    for i, seg in enumerate(segmentos):
        t = fn_modelo(seg)
        trad.append(t)
        print(f'  [{nome_mod}/{nome_tipo}] {i+1}/{len(segmentos)}: {t[:70]}')
    return trad

print('✓ Funções de geração definidas')

In [ ]:
# Célula 6 — Executar traduções (todos os modelos × todos os tipos)
# ⏱ ~20–40 min com GPU T4

TRADUCOES = {m: {} for m in MODELOS}

for nome_mod, info in MODELOS.items():
    for nome_tipo, corpus in CORPORA.items():
        print(f'\n{'='*55}\n  {nome_mod} → {nome_tipo}\n{'='*55}')
        TRADUCOES[nome_mod][nome_tipo] = traduzir_corpus(
            corpus['it'], info['fn'], nome_mod, nome_tipo
        )

print('\n✓ Todas as 16 traduções concluídas')

In [ ]:
# Célula 7 — TABELA 4: BLEU (Média ± DP amostral)

def bleu_corpus(traducoes, refs):
    return [bleu_sent.sentence_score(t, [r]).score
            for t, r in zip(traducoes, refs)]

RES_BLEU = {}
for m in MODELOS:
    RES_BLEU[m] = {}
    for nome_tipo, corpus in CORPORA.items():
        sc = bleu_corpus(TRADUCOES[m][nome_tipo], corpus['ref'])
        RES_BLEU[m][nome_tipo] = {
            'media': round(np.mean(sc), 2),
            'dp':    round(np.std(sc, ddof=1), 2),
            'scores': sc
        }

# Exibir tabela
linhas = []
for m in MODELOS:
    row = {'Modelo': m}
    for t in CORPORA:
        r = RES_BLEU[m][t]
        row[t] = f"{r['media']:.2f} ± {r['dp']:.2f}"
    linhas.append(row)
df4 = pd.DataFrame(linhas).set_index('Modelo')

print('\n' + '='*70)
print('TABELA 4 — BLEU por Modelo × Tipo Textual (Média ± DP amostral)')
print('='*70)
print(tabulate(df4, headers='keys', tablefmt='github'))

# Scores por segmento
print('\n--- Scores por segmento ---')
for m in MODELOS:
    for t in CORPORA:
        sc = RES_BLEU[m][t]['scores']
        print(f'  {m}/{t}: {[round(s,2) for s in sc]}')

In [ ]:
# Célula 8 — TABELA 5: IFC — Índice de Fidelidade Cultural

def calcular_ifc(traducoes, termos):
    pres, total = 0, 0
    for trad in traducoes:
        tl = trad.lower()
        for tp, ti in termos:
            total += 1
            if tp.lower() in tl or ti.lower() in tl:
                pres += 1
    return round((pres / total * 100) if total > 0 else 0.0, 1), pres, total

RES_IFC = {}
for m in MODELOS:
    RES_IFC[m] = {}
    for nome_tipo in CORPORA:
        pct, pres, tot = calcular_ifc(TRADUCOES[m][nome_tipo], TERMOS_IFC[nome_tipo])
        RES_IFC[m][nome_tipo] = {'ifc': pct, 'pres': pres, 'total': tot}

linhas_ifc = []
for m in MODELOS:
    row = {'Modelo': m}
    for t in CORPORA:
        row[t] = f"{RES_IFC[m][t]['ifc']:.1f}%"
    linhas_ifc.append(row)
df5 = pd.DataFrame(linhas_ifc).set_index('Modelo')

print('\n' + '='*70)
print('TABELA 5 — IFC: Índice de Fidelidade Cultural (%)')
print('='*70)
print(tabulate(df5, headers='keys', tablefmt='github'))

# Detalhe
print('\n--- Detalhe (presentes/total) ---')
for m in MODELOS:
    for t in CORPORA:
        r = RES_IFC[m][t]
        print(f'  {m}/{t}: {r["pres"]}/{r["total"]} = {r["ifc"]}%')

In [ ]:
# Célula 9 — TABELA 6: RTT — Round-Trip Translation BLEU
# Metodologia: IT → PT (modelo avaliado) → IT (M2M100) → BLEU vs original IT

print('Configurando retradução PT→IT com M2M100...')
tok_rtt = MODELOS['M2M100']['tok']
mod_rtt = MODELOS['M2M100']['mod']
tok_rtt.src_lang = 'pt'
lang_id_it = tok_rtt.get_lang_id('it')

def retraduir_it(texto_pt):
    enc = tok_rtt(texto_pt, return_tensors='pt', max_length=512, truncation=True).to(DEVICE)
    with torch.no_grad():
        out = mod_rtt.generate(**enc, forced_bos_token_id=lang_id_it, max_new_tokens=512)
    return tok_rtt.decode(out[0], skip_special_tokens=True)

RES_RTT = {}
for m in MODELOS:
    RES_RTT[m] = {}
    for nome_tipo, corpus in CORPORA.items():
        print(f'  RTT {m}/{nome_tipo}...')
        sc = []
        for pt, orig in zip(TRADUCOES[m][nome_tipo], corpus['it']):
            retrad = retraduir_it(pt)
            sc.append(bleu_sent.sentence_score(retrad, [orig]).score)
        RES_RTT[m][nome_tipo] = {
            'media': round(np.mean(sc), 2),
            'dp':    round(np.std(sc, ddof=1), 2),
            'scores': sc
        }

# Restaurar
tok_rtt.src_lang = 'it'

linhas_rtt = []
for m in MODELOS:
    row = {'Modelo': m}
    for t in CORPORA:
        r = RES_RTT[m][t]
        row[t] = f"{r['media']:.2f} ± {r['dp']:.2f}"
    linhas_rtt.append(row)
df6 = pd.DataFrame(linhas_rtt).set_index('Modelo')

print('\n' + '='*70)
print('TABELA 6 — RTT: Round-Trip Translation BLEU (Média ± DP amostral)')
print('='*70)
print(tabulate(df6, headers='keys', tablefmt='github'))

In [ ]:
# Célula 10 — Resumo executivo e exportação

print('='*70)
print('RESUMO EXECUTIVO — Tabelas 4, 5 e 6')
print('='*70)
print('\nTABELA 4 — BLEU (Média ± DP):')
print(df4.to_string())
print('\nTABELA 5 — IFC (%):')
print(df5.to_string())
print('\nTABELA 6 — RTT-BLEU (Média ± DP):')
print(df6.to_string())

# Melhor modelo por tipo e métrica
print('\n' + '-'*70)
print('Melhor por tipo textual:')
for t in CORPORA:
    mb  = max(MODELOS, key=lambda m: RES_BLEU[m][t]['media'])
    mi  = max(MODELOS, key=lambda m: RES_IFC[m][t]['ifc'])
    mr  = max(MODELOS, key=lambda m: RES_RTT[m][t]['media'])
    print(f'  {t:15} BLEU:{mb}({RES_BLEU[mb][t]["media"]:.2f})  IFC:{mi}({RES_IFC[mi][t]["ifc"]:.0f}%)  RTT:{mr}({RES_RTT[mr][t]["media"]:.2f})')

# Exportar Excel
with pd.ExcelWriter('tabelas_4_5_6_TCC_oficial.xlsx', engine='openpyxl') as w:
    df4.to_excel(w, sheet_name='Tabela4_BLEU')
    df5.to_excel(w, sheet_name='Tabela5_IFC')
    df6.to_excel(w, sheet_name='Tabela6_RTT')

print('\n✓ Exportado: tabelas_4_5_6_TCC_oficial.xlsx')
print('  (Baixe pelo painel de arquivos do Colab — ícone de pasta à esquerda)')

## Célula MHE — Mapa de Calor de Erros

Operacionaliza o instrumento da **seção 4.6** do TCC: alinhamento token a token entre tradução e referência, com classificação tipológica das divergências (omissão, substituição, inserção) adaptada de **Vilar et al. (2006)**.

**Rodar depois da Célula 6** (usa `TRADUCOES`, `CORPORA` e `MODELOS`). Gera 3 mapas de calor + tabela/CSV em `./figuras_mhe/`.

In [ ]:
# ============================================================================
# Célula MHE — Mapa de Calor de Erros
# Rodar DEPOIS da Célula 6 (precisa de TRADUCOES, CORPORA, MODELOS já definidos).
#
# Operacionaliza o instrumento descrito na seção 4.6 do TCC: alinhamento token a
# token entre tradução e referência, com classificação tipológica das divergências
# (omissão, substituição, inserção) adaptada de Vilar et al. (2006).
# ============================================================================
import os, re
import numpy as np
import pandas as pd
from difflib import SequenceMatcher
import matplotlib.pyplot as plt
import seaborn as sns

# Garantia: a célula depende das traduções reais geradas na Célula 6.
assert 'TRADUCOES' in globals(), "Rode a Célula 6 antes: TRADUCOES não existe."

os.makedirs('figuras_mhe', exist_ok=True)
CATS = ['omissao', 'substituicao', 'insercao']
CAT_LABEL = {'omissao': 'Omissão', 'substituicao': 'Substituição', 'insercao': 'Inserção'}


def _tok(t):
    """Tokenização simples: minúsculas, mantém acentos, ignora pontuação."""
    return re.findall(r"\w+", str(t).lower(), flags=re.UNICODE)


def categorizar_erros(ref, hyp):
    """Alinha referência (a) e tradução (b) por token e classifica as divergências.
       - delete  (token da referência sem correspondente na tradução) -> OMISSÃO
       - insert  (token gerado sem correspondente na referência)       -> INSERÇÃO
       - replace (bloco trocado)                                       -> SUBSTITUIÇÃO
    """
    a, b = _tok(ref), _tok(hyp)
    sm = SequenceMatcher(a=a, b=b, autojunk=False)
    c = {'correspondencia': 0, 'substituicao': 0, 'omissao': 0, 'insercao': 0}
    for op, i1, i2, j1, j2 in sm.get_opcodes():
        na, nb = i2 - i1, j2 - j1
        if op == 'equal':
            c['correspondencia'] += na
        elif op == 'delete':
            c['omissao'] += na
        elif op == 'insert':
            c['insercao'] += nb
        elif op == 'replace':
            m = min(na, nb)
            c['substituicao'] += m
            if na > nb:
                c['omissao'] += na - nb
            if nb > na:
                c['insercao'] += nb - na
    c['ref_tokens'] = max(len(a), 1)
    c['divergencias'] = c['substituicao'] + c['omissao'] + c['insercao']
    c['taxa_erro'] = min(c['divergencias'] / c['ref_tokens'], 1.0)  # clip p/ cor
    return c


# ---------------------------------------------------------------------------
# 1) Aplica a categorização a TODOS os segmentos: MHE[tipo][modelo] = [dict, ...]
# ---------------------------------------------------------------------------
MHE = {}
for tipo, corpus in CORPORA.items():
    MHE[tipo] = {}
    refs = corpus['ref']
    for m in MODELOS:
        hyps = TRADUCOES[m][tipo]
        if len(hyps) != len(refs):
            print(f"⚠ {m}/{tipo}: {len(hyps)} traduções vs {len(refs)} referências "
                  f"(usando o menor).")
        MHE[tipo][m] = [categorizar_erros(r, h) for r, h in zip(refs, hyps)]
print("✓ MHE calculado para", list(MHE.keys()))


# ---------------------------------------------------------------------------
# 2) HEAT MAP principal — Modelo × Segmento (taxa de erro por token), 1 por tipo
#    É o "mapa de calor" no sentido literal: hotspots vermelhos = onde o modelo quebra.
# ---------------------------------------------------------------------------
for tipo in CORPORA:
    mat = np.array([[seg['taxa_erro'] for seg in MHE[tipo][m]] for m in MODELOS])
    larg = max(8, mat.shape[1] * 0.55)
    plt.figure(figsize=(larg, 0.6 * len(MODELOS) + 1.5))
    sns.heatmap(mat, cmap='Reds', vmin=0, vmax=1, annot=True, fmt='.2f',
                linewidths=.5, linecolor='white',
                xticklabels=[f'S{i+1}' for i in range(mat.shape[1])],
                yticklabels=list(MODELOS),
                cbar_kws={'label': 'Taxa de erro por token (0–1)'})
    plt.title(f'Mapa de Calor de Erros — {tipo}', fontsize=12, weight='bold')
    plt.xlabel('Segmento'); plt.ylabel('Modelo')
    plt.tight_layout()
    plt.savefig(f'figuras_mhe/mhe_segmentos_{tipo}.png', dpi=150, bbox_inches='tight')
    plt.show()


# ---------------------------------------------------------------------------
# 3) HEAT MAP tipológico — Modelo × Categoria (omissão/substituição/inserção)
#    Valores normalizados pelo total de tokens de referência (proporção de erro por tipo).
# ---------------------------------------------------------------------------
linhas = []
for m in MODELOS:
    tot = {c: 0 for c in CATS}; reftok = 0
    for tipo in CORPORA:
        for seg in MHE[tipo][m]:
            for c in CATS:
                tot[c] += seg[c]
            reftok += seg['ref_tokens']
    linhas.append([tot[c] / reftok for c in CATS])
matc = np.array(linhas)
plt.figure(figsize=(6, 0.6 * len(MODELOS) + 1.5))
sns.heatmap(matc, cmap='Reds', annot=True, fmt='.3f', linewidths=.5, linecolor='white',
            xticklabels=[CAT_LABEL[c] for c in CATS], yticklabels=list(MODELOS),
            cbar_kws={'label': 'Erros / token de referência'})
plt.title('Perfil tipológico de erros (global)', fontsize=12, weight='bold')
plt.ylabel('Modelo')
plt.tight_layout()
plt.savefig('figuras_mhe/mhe_categorias_global.png', dpi=150, bbox_inches='tight')
plt.show()


# ---------------------------------------------------------------------------
# 4) HEAT MAP resumo — Modelo × Tipo textual (taxa de erro média)
# ---------------------------------------------------------------------------
matT = np.array([[np.mean([s['taxa_erro'] for s in MHE[tipo][m]]) for tipo in CORPORA]
                 for m in MODELOS])
plt.figure(figsize=(1.6 * len(CORPORA) + 2, 0.6 * len(MODELOS) + 1.5))
sns.heatmap(matT, cmap='Reds', vmin=0, vmax=1, annot=True, fmt='.2f',
            linewidths=.5, linecolor='white',
            xticklabels=list(CORPORA), yticklabels=list(MODELOS),
            cbar_kws={'label': 'Taxa de erro média por token'})
plt.title('Mapa de Calor de Erros — visão geral (Modelo × Tipo)', fontsize=12, weight='bold')
plt.ylabel('Modelo')
plt.tight_layout()
plt.savefig('figuras_mhe/mhe_resumo.png', dpi=150, bbox_inches='tight')
plt.show()


# ---------------------------------------------------------------------------
# 5) Tabela-resumo (para o texto / conferência) + export CSV
# ---------------------------------------------------------------------------
reg = []
for tipo in CORPORA:
    for m in MODELOS:
        d = {c: sum(s[c] for s in MHE[tipo][m]) for c in CATS}
        reftok = sum(s['ref_tokens'] for s in MHE[tipo][m])
        div = sum(d.values())
        reg.append({'Tipo': tipo, 'Modelo': m, **{CAT_LABEL[c]: d[c] for c in CATS},
                    'Tokens ref.': reftok, 'Taxa de erro': round(div / max(reftok, 1), 3)})
tab_mhe = pd.DataFrame(reg)
tab_mhe.to_csv('figuras_mhe/tabela_mhe.csv', index=False)
print('\nTABELA — Mapa de Calor de Erros (contagens por categoria)')
print(tab_mhe.to_string(index=False))
print('\n✓ Figuras e CSV salvos em ./figuras_mhe/')
